# SSM Occurence Indexer
```
ssm_occurrence{}
            |____ ssm{}
            |        |____ consequence[]
            |                     |_____ transcript{}
            |                                   |_____ gene{}
            |                                   |_____ annotation{}
            |____ case{}
                     |____ observation[]
```

In [1]:
import os
import requests
import uuid
%load_ext autoreload
from exports.mappings import GeneMapper, SSMMapper, Mapper
from exports.utils import get_array_paths

from pyspark.sql.functions import col, explode, collect_list, size, sum, first, struct, udf, regexp_extract, lit, count, broadcast
from pyspark.sql.types import StringType

## Load combined maf into spark

In [2]:
url = 's3a://test/smallest_mafs.csv'
    
df = sqlContext.read.format('com.databricks.spark.csv')\
                .options(header='true', inferschema='true')\
                .load(url)\
                .drop_duplicates()

In [3]:
#df = df.limit(500000)

## Rename and select desired columns in the mafs

In [4]:
%autoreload
from exports.utils import (
    maf_annotation_map,
    maf_gene_map,
    maf_observation_map,
    maf_ssm_map,
    maf_transcript_map,
    tumor_genotype_map,
    tumor_validation_map,
    normal_genotype_map,
    sample_map,
    input_bam_map,
    read_depth_map,
    maf_cols
)

maf_df = df.select(*( col(v).alias(k) for k,v in maf_cols.items() ))

## Augment maf df by extracting submitter_id and creating ssm_uuids

In [5]:
maf_df = maf_df.withColumn('_case_submitter_id',
                           regexp_extract(col('tumor_sample_barcode'),
                                          '([A-Z]{4}-[A-Z0-9]{2}-[A-Z0-9]{4})',1))
maf_ssm_map.update({'_case_submitter_id':'_case_submitter_id'})

In [6]:
def ssm_uuid(chromosome, start_position, ref_allele, tumor_allele):
    '''
    SNP: "{chromosome}:g.{start_position}{reference_allele}>{tumor_allele}"
    DEL: "{chromosome}:g.{start_position}del{reference_allele}"
    INS: "{chromosome}:g.{start_position}_{end_position}ins{tumor_allele}"
    '''
    chromosome = chromosome.replace('chr','')
    label = '{}:g.{}:{}>{}'.format(chromosome, start_position, ref_allele, tumor_allele)
    return str(uuid.uuid5(uuid.UUID('d15296a3-38ed-412e-8ace-75e235f82f55'), label))

ssm_uuid_udf =udf(ssm_uuid, StringType())
maf_df = maf_df.withColumn('ssm_id', ssm_uuid_udf(col('chromosome'), col('start_position'), col('reference_allele'), col('tumor_allele')))
maf_observation_map.update({'ssm_id':'ssm_id'})
maf_ssm_map.update({'ssm_id':'ssm_id'})

## Slice and dice until we get to the format we want

### SSM df

In [7]:
ssm_df = maf_df.select(*( col(k) for k in maf_ssm_map.keys() ))\
               .drop_duplicates()

## Transcript + Annotation + Gene
```
ssm{}
    |____ consequence[]
                |_____ transcript{}
                            |_____ gene{}
                            |_____ annotation{}
```

In [8]:
# Select annotation and gene into nested format
tran_df = maf_df.select('ssm_id', struct(
                                struct(*maf_annotation_map.keys()).alias('annotation'),
                                struct(*maf_gene_map.keys()).alias('gene'),
                                *set(maf_transcript_map.keys()) - set(['gene_id'])
                              ).alias('transcript'))

In [9]:
consequence_df = tran_df.select('ssm_id', struct('transcript').alias('consequence'))\
                        .groupBy('ssm_id')\
                        .agg(collect_list('consequence').alias('consequence'))

In [10]:
#consequence_df.printSchema()

In [11]:
ssm_tree = ssm_df.join(consequence_df, ssm_df.ssm_id == consequence_df.ssm_id, 'left')\
                    .drop(consequence_df.ssm_id)\
                    .select('_case_submitter_id',struct('consequence',*ssm_df.drop('_case_submitter_id').drop('gene_id').columns).alias('ssm'))

In [12]:
#ssm_tree.printSchema()

## Observation + Case + Occurance
```
occurrence[]
    |_____ case{}
             |____ observation[]
```

### Observation df

In [13]:
observation_df = maf_df.select('_case_submitter_id',
                               *(maf_observation_map.keys()
                                 +normal_genotype_map.keys()
                                 +tumor_genotype_map.keys()
                                 +tumor_validation_map.keys()
                                 +read_depth_map.keys()
                                 +input_bam_map.keys()
                                 +sample_map.keys()))

observation_df = observation_df.select('_case_submitter_id',
                                       struct(*normal_genotype_map.keys()).alias('normal_genotype'),
                                       struct(*tumor_genotype_map.keys()).alias('tumor_genotype'),
                                       struct(*tumor_validation_map.keys()).alias('validation'),
                                       struct(*read_depth_map.keys()).alias('read_depth'),
                                       struct(*input_bam_map.keys()).alias('input_bam_file'),
                                       struct(*sample_map.keys()).alias('sample'),
                                       *maf_observation_map.keys())\
                                .drop('ssm_id')\
                                .drop('gene_id')

In [14]:
#observation_df.count()
#observation_df.printSchema()

### Get case dataframe from existing graph

In [15]:
#doc = requests.get('http://elasticsearch.service.consul:9200/gdc_from_graph_35/_search').json()['hits']['hits'][0]['_source']
case_df = sqlContext.read.format("es")\
                    .option('es.nodes', 'elasticsearch.service.consul')\
                    .option('es.read.field.include', 'case_id,submitter_id,state,project.*,program.*,exposures.*,demographic.*')\
                    .option('es.resource.read', 'gdc_from_graph/case')\
                    .option('es.nodes.resolve.hostname','false')\
                    .load("gdc_from_graph")

In [16]:
# Merge observation with Case
ssm_occurrence_df = case_df.join(observation_df, case_df.submitter_id == observation_df._case_submitter_id, 'left')\
                        .select('submitter_id', struct(struct(observation_df.drop('_case_submitter_id').columns).alias('observation'),*case_df.columns).alias('case'))
        
ssm_occurrence = ssm_occurrence_df.join(ssm_tree, ssm_occurrence_df.submitter_id == ssm_tree._case_submitter_id)\
                    .drop('submitter_id')\
                    .drop('_case_submitter_id')
        


## Assemble constituent parts
```
ssm_occurrence{}
            |____ ssm{}
            |        |____ consequence[]
            |                     |_____ transcript{}
            |                                   |_____ gene{}
            |                                   |_____ annotation{}
            |____ case{}
                     |____ observation[]
```

ssm_occurrence = ssm_tree.join(occurence_df, ssm_tree._case_submitter_id == occurence_df.submitter_id, 'outer')\
                        .drop('submitter_id').drop('_case_submitter_id')

In [17]:
#ssm_occurrence.printSchema()

## Export df to es

In [18]:
sqlContext.sql("set spark.sql.shuffle.partitions=4096")

DataFrame[key: string, value: string]

In [19]:
index = 'dan-r1-ssm-occurrence'

In [44]:
%autoreload
from exports.mappings import SSMOccurrenceMapper
m = SSMOccurrenceMapper()

In [45]:
m.mapping['properties'].keys()
#m.mapping['_source'] = {'enabled':'true'}
m.mapping['_size'] =  {"enabled": 'true'}
#m.mapping['dynamic'] = 'true'
#m.mapping['properties']['ssm']['dynamic'] = 'true'
#m.mapping['properties']['gene']['properties']['ssm']['dynamic'] = 'true'
#m.mapping['properties']['gene']['properties']['ssm']['properties']['observation']['dynamic'] = 'true'
#m.mapping['properties']['case']['properties']['files']['properties']['cases']['dynamic'] = 'true'

In [46]:
import json

print requests.delete('http://elasticsearchvis.service.consul:9200/{}'.format(index)).json()

data = json.dumps({"settings":{"index":{
                "refresh_interval":"1m",
                "number_of_shards":20,
                "number_of_replicas":1,
                "mapper.dynamic":False,
                "mapping.nested_fields.limit":100,
                "mapping.total_fields.limit":2000
            }},"mappings":{
                "ssm-occurrence":m.mapping
            }})
#print requests.put('http://localhost:9200/test/', data=data).json()
print requests.put('http://elasticsearchvis.service.consul:9200/{}'.format(index), data=data).json()

{u'status': 404, u'error': {u'index_uuid': u'_na_', u'index': u'dan-r1-ssm-occurrence', u'resource.type': u'index_or_alias', u'root_cause': [{u'index_uuid': u'_na_', u'index': u'dan-r1-ssm-occurrence', u'resource.type': u'index_or_alias', u'resource.id': u'dan-r1-ssm-occurrence', u'reason': u'no such index', u'type': u'index_not_found_exception'}], u'reason': u'no such index', u'type': u'index_not_found_exception', u'resource.id': u'dan-r1-ssm-occurrence'}}
{u'acknowledged': True, u'shards_acknowledged': True}


In [47]:
#%%time
ssm_occurrence.write.format('org.elasticsearch.spark.sql')\
                    .option('es.nodes', 'elasticsearchvis.service.consul')\
                    .option('es.nodes.resolve.hostname','false')\
                    .option('es.resource.write', '{}/ssm-occurrence'.format(index))\
                    .option('es.http.timeout', '10m')\
                    .option('es.http.retries', '30')\
                    .option('es.batch.write.retry.count', '100')\
                    .option('es.batch.write.retry.wait', '10m')\
                    .option('es.batch.size.bytes','5mb')\
                    .option('es.batch.size.entries', '5000')\
                    .option('es.batch.write.refresh ', 'false')\
                    .save('{}/ssm-occurrence'.format(index))
            
#requests.post('http://localhost:9200/test-case/_refresh')

Py4JJavaError: An error occurred while calling o461.save.
: org.apache.spark.SparkException: Job 3 cancelled because Stage 11 was cancelled
	at org.apache.spark.scheduler.DAGScheduler.org$apache$spark$scheduler$DAGScheduler$$failJobAndIndependentStages(DAGScheduler.scala:1454)
	at org.apache.spark.scheduler.DAGScheduler.handleJobCancellation(DAGScheduler.scala:1393)
	at org.apache.spark.scheduler.DAGScheduler$$anonfun$handleStageCancellation$1.apply$mcVI$sp(DAGScheduler.scala:1381)
	at org.apache.spark.scheduler.DAGScheduler$$anonfun$handleStageCancellation$1.apply(DAGScheduler.scala:1380)
	at org.apache.spark.scheduler.DAGScheduler$$anonfun$handleStageCancellation$1.apply(DAGScheduler.scala:1380)
	at scala.collection.IndexedSeqOptimized$class.foreach(IndexedSeqOptimized.scala:33)
	at scala.collection.mutable.ArrayOps$ofInt.foreach(ArrayOps.scala:234)
	at org.apache.spark.scheduler.DAGScheduler.handleStageCancellation(DAGScheduler.scala:1380)
	at org.apache.spark.scheduler.DAGScheduler